In [ ]:
import gmsh

def mesh_gen(load_start, load_end, load_steps) :
    gmsh.initialize()
    gdim = 2
    model = gmsh.model.occ
    L_x = 1.0
    L_y = 1.0
    R_hole = 0.1
    mesh_size = 0.03
    # 2. Create rectangle
    rect = model.addRectangle(0.0, 0.0, 0.0, L_x, L_y)

    # 3. Create circle
    circle = model.addDisk(0.0, 0.0, 0.0, R_hole, R_hole)

    # 4. Cut the hole
    domain_with_hole, _ = model.cut([(gdim, rect)], [(gdim, circle)])
    gmsh.model.occ.synchronize()

    # 5. Physical group
    gmsh.model.addPhysicalGroup(gdim, [rect], 1)
    gmsh.model.setPhysicalName(gdim, 1, "domain")
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_size)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_size)
    gmsh.model.mesh.generate(gdim)
    # 6. Mesh generation
    gmsh.model.mesh.generate(gdim)

    # 7. Write to file
    gmsh.write("mesh.msh")
    gmsh.finalize()

In [ ]:
msh_data = read_from_msh("mesh_with_hole.msh", MPI.COMM_WORLD, 0, 2) 
domain = msh_data.mesh

node_coords = domain.geometry.x[:, :2]  # numpy array
num_nodes, gdim = node_coords.shape
# 3. Extract cell connectivity (triangles)
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, 0)
cells = domain.geometry.dofmap

np.savez("mesh/square_with_holes.npz", node_coords = node_coords, cells = cells)